# Private Markets Document Agent: Hybrid RAG + Multi-Agent Pipeline

**Business goal.** Turn heterogeneous financial documents (SEC 10-K filings, private-company board packs, undocumented Excel dumps) into a queryable knowledge base that supports both **exact quantitative answers** (SQL) and **semantic answers** (vector + keyword search), and finally produces an **investment-committee memo**.

**Pipeline overview**

| Phase | Purpose | Output |
|---|---|---|
| 1. Ingestion | Load public, private and orphan (header-less) tables | Standardized document dictionaries |
| 2. Processing | Normalize, validate, enrich with an LLM | `RAGReadyChunkSchema` objects |
| 3. Retrieval | DuckDB (SQL) + dense vectors + BM25, fused with RRF | Hybrid retriever |
| 4. Agents | Router, Analyst and Report agents orchestrated with LangGraph | Executive memo + risk flags |

**Design principle.** Numbers are never "recalled" by the LLM: exact figures come from SQL, while the LLM only interprets and narrates them. This minimizes hallucination risk in a regulated, audit-sensitive domain.

## Phase 1: Ingestion

### Step 0: Install dependencies
Installs `edgartools` (SEC XBRL access), `pandas`/`openpyxl` (tabular data and Excel I/O), `pydantic` (data contracts), `openai` (LLM + embeddings) and `tabulate` (needed by `DataFrame.to_markdown`). The `-q` flag keeps the output quiet.

In [ ]:
!pip install edgartools pandas openpyxl pydantic openai tabulate python-dotenv -q

### Step 1.1: Imports and working directory
Loads the standard stack and creates `./data/portfolio_ingestion`, where synthetic Excel files will be written. `mkdir(parents=True, exist_ok=True)` makes the cell **idempotent** (safe to re-run).

In [ ]:
# ============================================================
# STEP 1.1: Imports and working directory
# ============================================================

import os
import json
import uuid
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# Load secrets from the .env file (see .env.example)
load_dotenv()

DATA_DIR = Path("./data/portfolio_ingestion")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Dependencies and imports loaded successfully.")

### Step 1.2: Identity and API keys
- The SEC requires a descriptive `User-Agent` (name + email) for EDGAR requests; otherwise requests may be throttled or blocked.
- `OPENAI_API_KEY` is read by the OpenAI client from the environment.

All three values (`SEC_IDENTITY_NAME`, `SEC_IDENTITY_EMAIL`, `OPENAI_API_KEY`) are loaded from a local `.env` file via `python-dotenv`. Copy `.env.example` to `.env` and fill it in. `.env` is git-ignored, so secrets are never committed.

In [ ]:
# ============================================================
# STEP 1.2: Identity and API keys (read from .env)
# ============================================================
from edgar import set_identity, Company

# The SEC requires a User-Agent with a name and an email
SEC_IDENTITY_NAME = os.getenv("SEC_IDENTITY_NAME")
SEC_IDENTITY_EMAIL = os.getenv("SEC_IDENTITY_EMAIL")

if SEC_IDENTITY_NAME and SEC_IDENTITY_EMAIL:
    set_identity(f"{SEC_IDENTITY_NAME} {SEC_IDENTITY_EMAIL}")
    print("✅ SEC identity configured.")
else:
    raise RuntimeError("Set SEC_IDENTITY_NAME and SEC_IDENTITY_EMAIL in your .env file.")

if os.getenv("OPENAI_API_KEY"):
    print("✅ OpenAI API key detected.")
else:
    raise RuntimeError("Set OPENAI_API_KEY in your .env file before Step 1.4B / Phase 2.")

### Step 1.3: Public companies (SEC 10-K via XBRL)
Fetches the annual income statement from structured XBRL data rather than scraping PDFs/HTML, so values are machine-readable and comparable across issuers.

**Cleaning.** `dropna(how="all")` removes empty rows and `dropna(axis=1, how="all")` removes empty columns.

**Financial identities used downstream** (for revenue $R$, cost of goods sold $COGS$, operating expenses $OpEx$):

$$\text{Gross Profit} = R - COGS, \qquad \text{Gross Margin} = \frac{R - COGS}{R}$$

$$\text{Operating Income} = R - COGS - OpEx, \qquad \text{Operating Margin} = \frac{\text{Operating Income}}{R}$$

The function returns the standard **ingestion contract**: `source_type`, `company_name`, `ticker`, `document_title`, `dataframe`, `metadata`. Every later phase depends only on this contract, not on the data source.

In [ ]:
# ============================================================
# STEP 1.3: Public company ingestion (SEC 10-K via XBRL)
# ============================================================

def fetch_public_company_financials(ticker: str) -> Dict[str, Any]:
    """
    Fetches a clean income statement for a listed company using edgartools (XBRL).
    """
    print(f"🔍 [SEC Ingestor] Searching 10-K filings for {ticker}...")
    try:
        company = Company(ticker)
        financials = company.get_financials()
        income_stmt = financials.income_statement()
        df_income = income_stmt.to_dataframe()

        # Basic cleanup: drop fully empty rows/columns
        df_income = df_income.dropna(how="all").dropna(axis=1, how="all")

        print(f"  → Income statement retrieved for {company.name} ({df_income.shape[0]} rows, {df_income.shape[1]} columns)")

        return {
            "source_type": "public_sec_xbrl",
            "company_name": company.name,
            "ticker": ticker,
            "document_title": f"{ticker} Annual Report (10-K) - Income Statement",
            "dataframe": df_income,
            "metadata": {
                "cik": str(company.cik).zfill(10),
                "document_type": "10-K",
                "source": "SEC EDGAR XBRL"
            }
        }
    except Exception as e:
        print(f"❌ Error fetching data for {ticker}: {e}")
        return None

# Quick test with Apple
public_doc = fetch_public_company_financials("AAPL")
display(public_doc["dataframe"].head(10))

### Step 1.4: Private portfolio companies (Board Pack / Excel)
Private companies publish no XBRL, so data arrives as Excel board packs. Two sheets are simulated:

1. **Income Statement** by segment (values in USD thousands).
2. **SaaS KPIs** (ARR, NRR, margins by segment, CAC, LTV/CAC, headcount).

**Metric definitions** used in the data:

- Year-over-year growth, with $X_t$ the value in quarter $t$ and $X_{t-4}$ the same quarter one year earlier:
$$\text{YoY} = \frac{X_t}{X_{t-4}} - 1$$
- Gross profit and margin: $GP = R - COGS$, $GM = GP / R$.
- Adjusted EBITDA margin: $\text{EBITDA margin} = \dfrac{\text{EBITDA}}{R}$.
- Net Revenue Retention (starting recurring revenue $MRR_0$ of a cohort; expansion, contraction and churn measured over the period):
$$NRR = \frac{MRR_0 + \text{Expansion} - \text{Contraction} - \text{Churn}}{MRR_0}$$
An $NRR > 100\%$ means the existing customer base grows even without new logos.
- Unit economics:
$$\frac{LTV}{CAC}, \qquad LTV = \frac{ARPA \times GM}{\text{churn rate}}$$
A ratio above roughly $3\times$ is a common healthy benchmark.

**Sanity check on the mock data (Q3 2024):** $6100 + 2050 = 8150$ total revenue and $8150 - 1467 = 6683$ gross profit, so the sheet is internally consistent.

Segment mix matters: Enterprise grows (+48% YoY) while SMB shrinks (-10%), so total growth hides diverging dynamics.

In [ ]:
# ============================================================
# STEP 1.4: Private portfolio company ingestion (Board Deck / Excel)
# ============================================================

def generate_mock_portfolio_board_pack() -> Path:
    """
    Generates a synthetic but realistic Excel file that simulates a
    'Q3 Board Deck / Financial Model' of a Private Equity / VC portfolio company.
    """
    excel_path = DATA_DIR / "SaaS_Portfolio_Co_Q3_Board_Pack.xlsx"

    # 1. P&L / Income Statement by segment
    pnl_data = {
        "Metric / Segment": [
            "Revenue: Enterprise Segment",
            "Revenue: SMB Segment",
            "Total Net Revenue",
            "Cost of Goods Sold (COGS)",
            "Gross Profit",
            "Operating Expenses (OpEx): R&D",
            "Operating Expenses (OpEx): S&M",
            "Operating Expenses (OpEx): G&A",
            "EBITDA Adjusted",
            "Net Profit / (Loss)"
        ],
        "Q4 2023 ($K)": [4100, 2300, 6400, -1280, 5120, -1800, -2200, -900, 220, -150],
        "Q1 2024 ($K)": [4500, 2250, 6750, -1350, 5400, -1900, -2300, -920, 280, -100],
        "Q2 2024 ($K)": [5200, 2100, 7300, -1387, 5913, -2000, -2450, -950, 513, 100],
        "Q3 2024 ($K)": [6100, 2050, 8150, -1467, 6683, -2100, -2600, -980, 1003, 450],
        "YoY Growth (%)": [0.48, -0.10, 0.27, 0.14, 0.30, 0.16, 0.18, 0.08, 3.55, 4.00]
    }

    # 2. Operating SaaS / Private Markets KPIs
    kpi_data = {
        "SaaS Metric": [
            "Annual Recurring Revenue (ARR)",
            "Net Revenue Retention (NRR %)",
            "Gross Margin Enterprise (%)",
            "Gross Margin SMB (%)",
            "Customer Acquisition Cost (CAC - USD)",
            "LTV / CAC Ratio",
            "Headcount (FTEs)"
        ],
        "Q3 2023": ["$22.5M", "108%", "82.0%", "68.0%", "$14,500", "3.2x", "110"],
        "Q4 2023": ["$24.2M", "110%", "82.5%", "67.5%", "$14,200", "3.4x", "115"],
        "Q1 2024": ["$25.8M", "111%", "83.0%", "66.0%", "$13,800", "3.6x", "120"],
        "Q2 2024": ["$28.1M", "113%", "84.1%", "65.2%", "$13,500", "3.9x", "128"],
        "Q3 2024": ["$31.4M", "115%", "85.2%", "64.8%", "$13,100", "4.2x", "135"]
    }

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        pd.DataFrame(pnl_data).to_excel(writer, sheet_name="Income Statement", index=False)
        pd.DataFrame(kpi_data).to_excel(writer, sheet_name="SaaS KPIs", index=False)

    print(f"📁 Synthetic Board Pack saved to: {excel_path}")
    return excel_path


def load_private_portfolio_document(excel_path: Path, sheet_name: str = "Income Statement") -> Dict[str, Any]:
    """
    Parses private-company spreadsheets / Board Packs into clean DataFrames.
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    company_name = "CloudScale Tech (Portfolio Co #4)"

    print(f"🔍 [Private Portfolio Ingestor] Loading sheet '{sheet_name}' for {company_name}...")

    return {
        "source_type": "private_portfolio_excel",
        "company_name": company_name,
        "ticker": "PORTFOLIO_CO_04",
        "document_title": f"{company_name} - Q3 2024 Board Deck ({sheet_name})",
        "dataframe": df,
        "metadata": {
            "fund_id": "PE_FUND_III",
            "document_type": "Board Pack / Financial Model",
            "sheet_name": sheet_name,
            "currency": "USD (Thousands for P&L)",
            "source": "Portfolio Portal Upload"
        }
    }

# Generate and ingest the test Board Pack
mock_file = generate_mock_portfolio_board_pack()

private_doc_pnl = load_private_portfolio_document(mock_file, sheet_name="Income Statement")
private_doc_kpis = load_private_portfolio_document(mock_file, sheet_name="SaaS KPIs")

print("\n--- Preview: Board Pack Income Statement ---")
display(private_doc_pnl["dataframe"])

print("\n--- Preview: Board Pack SaaS KPIs ---")
display(private_doc_kpis["dataframe"])

### Step 1.4B (Part 1): Auto-documenter for orphan tables
**Problem.** Portfolio companies often send raw dumps with no headers, title, currency or period, only columns `0, 1, 2, ...`. Such tables cannot be indexed or compared.

**Approach.** Send a 10-row sample plus any surrounding text to an LLM at `temperature=0.0` (deterministic, reproducible) and force a JSON response. The model reconstructs the title, company, column headers, currency, period and a `confidence_score` $c \in [0, 1]$.

**Safeguards.**
- The reconstructed headers are applied only if their count equals the column count:
$$|\text{inferred\_headers}| = |\text{columns}|$$
otherwise generic names are used. This prevents silent column misalignment.
- Inference is probabilistic, so the result is flagged in metadata (`is_inferred_schema`, `inference_confidence`) for **auditability**.

The mock orphan table has rows as metrics (ARR, NRR, CAC payback, Burn Multiple, Rule of 40) and columns as periods.

**Metrics contained in the orphan table:**
$$\text{Burn Multiple} = \frac{\text{Net Burn}}{\text{Net New ARR}} \quad (\text{lower is better; } <1.5 \text{ is typically efficient})$$
$$\text{CAC Payback (months)} = \frac{CAC}{ARPA_{\text{monthly}} \times GM}$$
$$\text{Rule of 40} = \text{Revenue Growth \%} + \text{Profit Margin \%} \quad (\ge 40\% \text{ is healthy})$$

In [ ]:
# ============================================================
# STEP 1.4B (Part 1): Auto-documenter and orphan table inspector
# ============================================================

from typing import Tuple

def auto_document_orphan_table(
    df: pd.DataFrame,
    surrounding_text: str = "",
    source_filename: str = "Unknown_Document.xlsx"
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Takes an undocumented table (generic columns 0, 1, 2...), analyzes its content
    and surrounding text, and infers headers, title, currency and main metrics
    using gpt-4o-mini.
    """
    import openai

    sample_table_str = df.head(10).to_string()

    prompt = f"""
You are an expert in financial parsing for Private Equity / Venture Capital.
An ORPHAN or UNDOCUMENTED financial table (unlabeled or generic columns) was detected in the file "{source_filename}".

Surrounding text / sheet name (if any):
"{surrounding_text}"

Raw table sample:
{sample_table_str}

Your task is to analyze the values and patterns to RECONSTRUCT and DOCUMENT the table:
1. "inferred_title": A reasonable descriptive title (e.g. "SaaS KPIs & Efficiency Metrics").
2. "inferred_company_name": Likely company name or code (if none, use "Apex Analytics (Portfolio Co #7)").
3. "inferred_headers": A list of reconstructed column names based on the row contents.
4. "inferred_currency": Likely currency ("USD", "EUR", "Unknown").
5. "inferred_period": Detected period or years (e.g. "Q1-Q4 2024").
6. "confidence_score": A float from 0.0 to 1.0 indicating your confidence.

Return strictly a JSON with this structure:
{{
  "inferred_title": "...",
  "inferred_company_name": "...",
  "inferred_headers": ["Col1", "Col2", "..."],
  "inferred_currency": "...",
  "inferred_period": "...",
  "confidence_score": 0.90
}}
"""

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    inferred_info = json.loads(response.choices[0].message.content)

    # Rebuild the DataFrame with the new headers
    df_reconstructed = df.copy()
    new_headers = inferred_info.get("inferred_headers", [])

    if len(new_headers) == len(df_reconstructed.columns):
        df_reconstructed.columns = new_headers
    else:
        cols = [f"Col_{i}" for i in range(len(df_reconstructed.columns))]
        cols[0] = "Metric / Item (Inferred)"
        df_reconstructed.columns = cols

    return df_reconstructed, inferred_info


def generate_orphan_portfolio_excel() -> Path:
    """
    Generates a 'dirty' Excel file with no column names or metadata,
    simulating a raw dump received by email from a portfolio company.
    """
    orphan_path = DATA_DIR / "Portfolio_Co_07_Raw_Dump.xlsx"

    # Raw data without descriptive headers (columns 0, 1, 2, 3)
    raw_dirty_data = {
        0: ["ARR Cloud Services ($M)", "Net Revenue Retention (NRR)", "CAC Payback (Months)", "Burn Multiple", "Rule of 40 Score"],
        1: ["12.4", "112%", "14.2", "1.1x", "42%"],
        2: ["15.8", "115%", "12.8", "0.9x", "48%"],
        3: ["18.5", "118%", "11.5", "0.7x", "54%"]
    }

    df_raw = pd.DataFrame(raw_dirty_data)

    # Save without headers (header=False)
    df_raw.to_excel(orphan_path, sheet_name="Sheet1_Dump", index=False, header=False)
    print(f"📁 Orphan / unstructured Excel saved to: {orphan_path}")
    return orphan_path

print("✅ Orphan table detection module loaded.")

### Step 1.4B (Part 2): Orphan document loader
Reads the sheet with `header=None` (so row 0 is not mistaken for headers), calls the auto-documenter, and wraps the result in the **same ingestion contract** as the other sources.

Audit flags are stored in `metadata`:
- `is_undocumented_source = True`
- `is_inferred_schema = True`
- `inference_confidence` $= c$

Downstream, the Report Agent uses these flags to add an audit warning. A reasonable operating policy is to route documents with $c < \tau$ (for example $\tau = 0.8$) to human review.

In [ ]:
# ============================================================
# STEP 1.4B (Part 2): Orphan document loader and ingestor
# ============================================================

def load_orphan_portfolio_document(excel_path: Path, sheet_name: str = "Sheet1_Dump") -> Dict[str, Any]:
    """
    Parses an unstructured Excel table, runs LLM auto-documentation,
    and adapts it to the standard ingestion contract with audit flags.
    """
    print(f"🔍 [Orphan Ingestor] Unstructured table detected in '{excel_path.name}' ({sheet_name})...")

    # Read without headers
    df_raw = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)

    # Auto-document and infer
    df_clean, inferred_meta = auto_document_orphan_table(
        df=df_raw,
        surrounding_text=f"Sheet '{sheet_name}' in a raw file received by email.",
        source_filename=excel_path.name
    )

    company_name = inferred_meta.get("inferred_company_name", "Apex Analytics (Portfolio Co #7)")
    doc_title = f"{company_name} - {inferred_meta.get('inferred_title', 'Unstructured Report')} (Auto-Documented)"

    return {
        "source_type": "private_orphan_excel",
        "company_name": company_name,
        "ticker": "PORTFOLIO_CO_07",
        "document_title": doc_title,
        "dataframe": df_clean,
        "metadata": {
            "fund_id": "PE_FUND_III",
            "document_type": "Unstructured Raw Dump",
            "sheet_name": sheet_name,
            "currency": inferred_meta.get("inferred_currency", "USD"),
            "source": "Unstructured Email Dump",
            "is_undocumented_source": True,
            "is_inferred_schema": True,
            "inference_confidence": inferred_meta.get("confidence_score", 0.90),
            "inferred_period": inferred_meta.get("inferred_period", "Q1-Q3 2024")
        }
    }


# Test generation and loading of the orphan file
orphan_mock_file = generate_orphan_portfolio_excel()
orphan_doc = load_orphan_portfolio_document(orphan_mock_file)

print("\n--- Preview: Automatically Reconstructed Orphan Table ---")
print(f"📌 Reconstructed title: {orphan_doc['document_title']}")
print(f"📌 Inference confidence: {orphan_doc['metadata']['inference_confidence'] * 100}%")
display(orphan_doc["dataframe"])

### Step 1.5: Unified Ingestion Agent
A facade class exposing one entry point per source type (`ingest_public_ticker`, `ingest_private_board_pack`, `ingest_orphan_document`) and storing all results in `loaded_documents`.

**Why:** a single, uniform list of documents decouples Phase 2 from the origin of the data. Failures are caught per document or sheet, so one bad file does not abort the batch.

If $N_{pub}$ public tickers, $S$ board-pack sheets and $N_{orph}$ orphan files are loaded, the expected document count is:
$$N = N_{pub} + S + N_{orph} = 2 + 2 + 1 = 5$$

In [ ]:
# ============================================================
# STEP 1.5: Unified Ingestion Agent (public + private + orphan)
# ============================================================

class UnifiedIngestionAgent:
    """
    Unified ingestion agent able to process SEC reports (10-K),
    structured Board Packs and orphan / undocumented Excel files.
    """
    def __init__(self):
        self.loaded_documents: List[Dict[str, Any]] = []

    def ingest_public_ticker(self, ticker: str) -> Optional[Dict[str, Any]]:
        doc = fetch_public_company_financials(ticker)
        if doc:
            self.loaded_documents.append(doc)
        return doc

    def ingest_private_board_pack(self, excel_path: Path) -> List[Dict[str, Any]]:
        docs = []
        for sheet in ["Income Statement", "SaaS KPIs"]:
            try:
                doc = load_private_portfolio_document(excel_path, sheet_name=sheet)
                self.loaded_documents.append(doc)
                docs.append(doc)
            except Exception as e:
                print(f"⚠️ Could not load sheet {sheet}: {e}")
        return docs

    def ingest_orphan_document(self, excel_path: Path) -> Optional[Dict[str, Any]]:
        try:
            doc = load_orphan_portfolio_document(excel_path)
            self.loaded_documents.append(doc)
            return doc
        except Exception as e:
            print(f"⚠️ Could not process orphan file {excel_path}: {e}")
            return None

    def summary(self):
        print("\n" + "="*70)
        print("INGESTED DOCUMENTS SUMMARY (STEP 1 - UNIFIED)")
        print("="*70)
        for idx, d in enumerate(self.loaded_documents):
            is_inferred = d['metadata'].get('is_inferred_schema', False)
            tag = " [AUTO-DOCUMENTED]" if is_inferred else ""
            print(f"[{idx+1}] {d['company_name']} ({d['ticker']}){tag}")
            print(f"    │ Type: {d['source_type']}")
            print(f"    │ Title: {d['document_title']}")
            print(f"    └ Shape: {d['dataframe'].shape}")

# Initialize and run the full ingestion
ingestion_agent = UnifiedIngestionAgent()

# 1. Ingest 2 public companies (SEC 10-K)
ingestion_agent.ingest_public_ticker("AAPL")
ingestion_agent.ingest_public_ticker("MSFT")

# 2. Ingest 1 private portfolio company (standard Board Pack)
ingestion_agent.ingest_private_board_pack(mock_file)

# 3. Ingest 1 undocumented private company (orphan table)
ingestion_agent.ingest_orphan_document(orphan_mock_file)

# Show global summary
ingestion_agent.summary()

## Phase 2: Processing

Each ingested table is converted into a **RAG-ready chunk** carrying three synchronized representations:

| Field | Used for | Why |
|---|---|---|
| `vectorContent` | Embeddings (semantic search) | Narrative plus context, optimized for meaning |
| `rawContent` | Exact context given to the LLM | Clean Markdown table |
| `structuredTable` | DuckDB / SQL | Exact numeric answers |

### Step 2.1: Cleaning, normalization and format converters
Three converters:

1. `clean_and_normalize_df`: casts column names to stripped strings and replaces `NaN` with `None`, because JSON has no `NaN`.
2. `df_to_canonical_json`: produces a canonical record-oriented schema (`columns`, `primary_key_col`, `rows`, `currency`). NumPy scalars are cast to native Python types for serialization.
3. `df_to_clean_markdown`: renders at most `max_rows` rows as Markdown, formatting numbers as `1,234.56`, to keep the LLM prompt bounded.

**Token budget.** The prompt size grows roughly linearly with rows $r$ and columns $k$:
$$T_{\text{prompt}} \approx \alpha \cdot r \cdot k + T_{\text{instructions}}$$
Capping $r \le 35$ bounds both cost and latency.

In [ ]:
# ============================================================
# STEP 2.1: Cleaning, normalization and format converters
# ============================================================

def clean_and_normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizes the DataFrame: cleans column names, null values and numeric formats.
    """
    df = df.copy()

    # 1. Ensure column names are clean strings
    df.columns = [str(c).strip() for c in df.columns]

    # 2. Replace null/NaN with None for JSON compatibility
    df = df.where(pd.notnull(df), None)

    return df


def df_to_canonical_json(df: pd.DataFrame, table_name: str, currency: str = "USD") -> Dict[str, Any]:
    """
    Converts the DataFrame into a canonical JSON schema ready to store in DuckDB / SQLite.
    """
    df_clean = clean_and_normalize_df(df)

    # Identify the main label/metric column
    first_col = df_clean.columns[0]

    records = []
    for _, row in df_clean.iterrows():
        row_dict = {}
        for col in df_clean.columns:
            val = row[col]
            # Cast np.number to native Python types for JSON
            if isinstance(val, (np.integer, np.floating)):
                val = float(val) if isinstance(val, np.floating) else int(val)
            row_dict[col] = val
        records.append(row_dict)

    return {
        "table_name": table_name,
        "columns": list(df_clean.columns),
        "primary_key_col": first_col,
        "rows": records,
        "currency": currency
    }


def df_to_clean_markdown(df: pd.DataFrame, max_rows: int = 35) -> str:
    """
    Converts the DataFrame into a readable, well-formatted Markdown table for the LLM.
    """
    df_clean = clean_and_normalize_df(df).head(max_rows)

    # Visual formatting of numbers
    df_formatted = df_clean.copy()
    for col in df_formatted.columns:
        df_formatted[col] = df_formatted[col].apply(
            lambda x: f"{x:,.2f}" if isinstance(x, (float, int)) and not isinstance(x, bool) else (x if x is not None else "—")
        )

    return df_formatted.to_markdown(index=False)

print("✅ Normalization and conversion functions ready.")

### Step 2.2: Strict data contract (Pydantic)
Defines typed schemas so malformed objects fail fast, at the system boundary, instead of corrupting the index later:

- `StructuredTableSchema`: canonical table for SQL.
- `ChunkMetadataSchema`: filterable metadata (company, ticker, source type, KPIs, segments).
- `RAGReadyChunkSchema`: the full chunk (`id`, `vectorContent`, `rawContent`, `structuredTable`, `metadata`).

Formally, each chunk is a tuple $c = (id, v, r, T, m)$ where $v$ is embedded, $r$ is shown verbatim, $T$ is queried by SQL, and $m$ is used for filtering. Keeping $r$ and $T$ separate from $v$ ensures the narrative never overrides the source numbers.

In [ ]:
# ============================================================
# STEP 2.2: Strict data contract (Pydantic schema)
# ============================================================

class StructuredTableSchema(BaseModel):
    table_name: str
    columns: List[str]
    primary_key_col: str
    rows: List[Dict[str, Any]]
    currency: str = "USD"

class ChunkMetadataSchema(BaseModel):
    companyName: str
    tickerOrCode: str
    documentTitle: str
    sourceType: str  # 'public_sec_xbrl' | 'private_portfolio_excel'
    documentType: str
    currency: str
    kpisPresent: List[str] = Field(default_factory=list)
    segmentsPresent: List[str] = Field(default_factory=list)
    enrichmentModel: str = "gpt-4o-mini"

class RAGReadyChunkSchema(BaseModel):
    id: str
    vectorContent: str          # Enriched content → embedding (vector DB)
    rawContent: str             # Clean Markdown table → exact context for the LLM
    structuredTable: StructuredTableSchema  # Canonical JSON → DuckDB / SQL
    metadata: ChunkMetadataSchema

print("✅ RAGReadyChunkSchema Pydantic schema defined.")

### Step 2.3: LLM enrichment (table-to-narrative + contextual chunking)
**Problem.** Embedding a raw numeric table works poorly: numbers carry little semantic signal, so a query like "how did margin evolve?" is far from a grid of digits in vector space.

**Solution.**
1. The LLM writes a **narrative** (trends, YoY/QoQ comparisons, segments), extracts **KPIs** and **segments**, and lists **key insights**.
2. **Contextual chunking** (Anthropic style): prepend global context (company, source, document, currency) to the chunk so it is self-explanatory when retrieved in isolation.

**Quantities the narrative is expected to describe:**
$$\text{QoQ} = \frac{X_t}{X_{t-1}} - 1, \qquad \text{YoY} = \frac{X_t}{X_{t-4}} - 1$$
$$\text{Margin}_t = \frac{\text{Profit}_t}{\text{Revenue}_t}, \qquad \Delta_{pp} = \text{Margin}_t - \text{Margin}_{t-4} \ \text{(percentage points)}$$

`temperature=0.1` keeps the output near-deterministic while allowing fluent prose. The original Markdown table is appended at the end of `vectorContent`, so the embedded text contains both the interpretation and the source evidence.

In [ ]:
# ============================================================
# STEP 2.3: LLM enrichment (table-to-narrative + contextual chunking)
# ============================================================

def enrich_document_for_rag(doc_item: Dict[str, Any]) -> RAGReadyChunkSchema:
    """
    Takes a document ingested in Step 1, runs LLM narrative synthesis,
    and returns a validated RAGReadyChunk object.
    """
    company_name = doc_item["company_name"]
    ticker = doc_item["ticker"]
    doc_title = doc_item["document_title"]
    source_type = doc_item["source_type"]
    df = doc_item["dataframe"]

    markdown_table = df_to_clean_markdown(df)
    canonical_json = df_to_canonical_json(df, table_name=doc_title)

    # Pre-RAG financial analysis prompt
    prompt = f"""
You are a Senior Financial Analyst at a Private Equity / Venture Capital firm.

Analyze the following table extracted from the document "{doc_title}" of the company "{company_name}" ({ticker}).

Table in Markdown:
{markdown_table}

Respond in JSON format strictly with the following keys:
1. "narrative": A fluent, detailed analytical summary in English describing the evolution of key metrics, time comparisons (YoY/QoQ), margins or growth by segment. It must be optimized for semantic search.
2. "kpis": List of detected financial or operating KPIs (e.g. ["Revenue", "Gross Margin", "EBITDA", "ARR", "NRR"]).
3. "segments": List of business segments or divisions mentioned (e.g. ["Enterprise", "SMB", "Products", "Services"]). If there are no segments, an empty list [].
4. "key_insights": 2-4 key numeric findings or quantitative trends.
"""

    import openai
    client = openai.OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    llm_output = json.loads(response.choices[0].message.content)

    # === CONTEXTUAL CHUNKING (Anthropic-style) ===
    vector_content = f"""
[GLOBAL DOCUMENT CONTEXT]
Company: {company_name} ({ticker})
Source Type: {source_type}
Document: {doc_title}
Currency: {doc_item.get('metadata', {}).get('currency', 'USD')}

[DETECTED KPIs]: {', '.join(llm_output.get('kpis', []))}
[SEGMENTS]: {', '.join(llm_output.get('segments', []))}

[TABLE NARRATIVE SUMMARY]:
{llm_output.get('narrative', '')}

[KEY INSIGHTS]:
{chr(10).join('• ' + item for item in llm_output.get('key_insights', []))}

[ORIGINAL TABLE IN MARKDOWN]:
{markdown_table}
""".strip()

    chunk_id = str(uuid.uuid4())

    metadata = ChunkMetadataSchema(
        companyName=company_name,
        tickerOrCode=ticker,
        documentTitle=doc_title,
        sourceType=source_type,
        documentType=doc_item.get("metadata", {}).get("document_type", "Financial Report"),
        currency=doc_item.get("metadata", {}).get("currency", "USD"),
        kpisPresent=llm_output.get("kpis", []),
        segmentsPresent=llm_output.get("segments", []),
        enrichmentModel="gpt-4o-mini"
    )

    return RAGReadyChunkSchema(
        id=chunk_id,
        vectorContent=vector_content,
        rawContent=markdown_table,
        structuredTable=StructuredTableSchema(**canonical_json),
        metadata=metadata
    )

print("✅ enrich_document_for_rag configured with gpt-4o-mini.")

### Step 2.4: Batch execution
Iterates over `ingestion_agent.loaded_documents`, enriching each one. Each document is wrapped in `try/except`, so a single API or validation failure does not discard the whole batch.

Success rate:
$$\text{Coverage} = \frac{|\text{rag\_ready\_chunks}|}{N}$$
Ideally $\text{Coverage} = 1$. Anything lower should be investigated, since missing chunks silently reduce what the agents can answer.

Cost scales linearly: $\text{Cost} \approx N \cdot (\text{tokens}_{in} \cdot p_{in} + \text{tokens}_{out} \cdot p_{out})$.

In [ ]:
# ============================================================
# STEP 2.4: Batch execution over ingestion_agent.loaded_documents
# ============================================================

rag_ready_chunks: List[RAGReadyChunkSchema] = []
total_docs = len(ingestion_agent.loaded_documents)

print(f"🚀 Processing and enriching the {total_docs} documents ingested in Step 1...\n")

for idx, doc in enumerate(ingestion_agent.loaded_documents):
    print(f"[{idx+1}/{total_docs}] Processing: {doc['company_name']} - {doc['document_title']}...")
    try:
        chunk = enrich_document_for_rag(doc)
        rag_ready_chunks.append(chunk)
        print(f"    ✓ RAG-ready object generated successfully (ID: {chunk.id[:8]}...)")
        print(f"      Extracted KPIs: {chunk.metadata.kpisPresent}")
        print(f"      Extracted segments: {chunk.metadata.segmentsPresent}\n")
    except Exception as e:
        print(f"    ❌ Error processing {doc['company_name']}: {e}\n")

print(f"🎉 Processing complete! Total RAG-ready chunks: {len(rag_ready_chunks)}")

### Step 2.5: Inspecting a RAG-ready chunk
Manual quality-assurance step. It prints the three views of one chunk (CloudScale Tech):

1. `vectorContent`: what gets embedded.
2. `metadata`: structured filters for hybrid search.
3. `structuredTable`: the first two canonical rows destined for SQL.

Review whether the narrative numbers agree with the table. For example, the Q3 2024 EBITDA margin should be about $1003 / 8150 \approx 12.3\%$, up from $220 / 6400 \approx 3.4\%$ in Q4 2023.

In [ ]:
# ============================================================
# STEP 2.5: Inspecting a RAG-ready chunk (private portfolio example)
# ============================================================

# Find the CloudScale Tech chunk (Board Pack)
portfolio_chunk = next(c for c in rag_ready_chunks if c.metadata.tickerOrCode == "PORTFOLIO_CO_04")

print("="*70)
print("RAG-READY CHUNK EXAMPLE (CloudScale Tech - Private Portfolio)")
print("="*70)

print("\n1️⃣ --- vectorContent (text embedded in the vector store) ---")
print(portfolio_chunk.vectorContent[:1500] + "\n\n[...rest of vectorContent...]")

print("\n2️⃣ --- metadata (structured filters for hybrid search) ---")
print(json.dumps(portfolio_chunk.metadata.model_dump(), indent=2, ensure_ascii=False))

print("\n3️⃣ --- structuredTable (canonical schema for DuckDB/SQL - first 2 rows) ---")
table_preview = portfolio_chunk.structuredTable.model_dump()
table_preview["rows"] = table_preview["rows"][:2]
print(json.dumps(table_preview, indent=2, ensure_ascii=False))

## Phase 3: Hybrid retrieval

Neither retrieval paradigm is sufficient alone:

| Method | Strength | Weakness |
|---|---|---|
| SQL (DuckDB) | Exact numbers | Needs a precise question |
| Dense vectors | Semantic similarity | Can miss exact terms such as "NRR" |
| BM25 (sparse) | Exact keyword match | No synonym understanding |

They are combined and fused with Reciprocal Rank Fusion.

### Step 3.1: Install DuckDB and BM25
`duckdb` is an embedded analytical SQL engine (no server needed). `rank-bm25` implements the Okapi BM25 ranking function.

In [ ]:
# ============================================================
# STEP 3.1: Install DuckDB and BM25
# ============================================================
!pip install duckdb rank-bm25 numpy -q

import duckdb
import numpy as np
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any, Tuple

print("✅ DuckDB and Rank-BM25 installed successfully.")

### Step 3.2: DuckDB structured store
Registers each canonical table in an in-memory DuckDB database so quantitative questions are answered with **deterministic SQL**, not LLM recall.

Each table gets a safe name `{ticker}_{pnl|kpis}`, derived from the ticker and the document title.

**Why SQL for numbers.** For an aggregate such as the total of a column $x$ over rows $i$:
$$S = \sum_{i=1}^{n} x_i$$
the database returns an exact result, whereas an LLM may approximate or invent it. Relational algebra also supports filters ($\sigma$) and projections ($\pi$), for example $\pi_{\text{metric},\text{Q3}}(\sigma_{\text{metric LIKE '\%EBITDA\%'}}(T))$.

**Security note:** table names are built from internal metadata. If they ever come from user input, they must be validated/whitelisted to prevent SQL injection.

In [ ]:
# ============================================================
# STEP 3.2: DuckDB initialization and table registration
# ============================================================

class DuckDBStructuredStore:
    """
    Stores the canonical tables in an in-memory DuckDB database
    to allow exact quantitative SQL queries without hallucinations.
    """
    def __init__(self):
        self.con = duckdb.connect(database=':memory:')
        self.table_names: List[str] = []

    def register_chunk_tables(self, chunks: List[RAGReadyChunkSchema]):
        for chunk in chunks:
            # Build a safe SQL table name
            safe_ticker = chunk.metadata.tickerOrCode.lower().replace("-", "_").replace(" ", "_")
            doc_type = "pnl" if "income" in chunk.metadata.documentTitle.lower() or "statement" in chunk.metadata.documentTitle.lower() else "kpis"
            table_name = f"{safe_ticker}_{doc_type}"

            # Convert canonical JSON rows to a Pandas DataFrame for DuckDB
            rows = chunk.structuredTable.rows
            if not rows:
                continue

            df = pd.DataFrame(rows)

            # Register in DuckDB
            self.con.register(table_name, df)

            # Create a physical in-memory table
            self.con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM {table_name}")
            self.table_names.append(table_name)

            print(f"  ✓ SQL table registered in DuckDB: '{table_name}' ({df.shape[0]} rows, {df.shape[1]} cols)")

    def query_sql(self, sql_query: str) -> pd.DataFrame:
        """Runs a free-form SQL query over the registered tables."""
        try:
            return self.con.execute(sql_query).df()
        except Exception as e:
            print(f"❌ Error running SQL: {e}")
            return pd.DataFrame()

    def list_tables(self) -> List[str]:
        return self.table_names

# Initialize and register the tables
duckdb_store = DuckDBStructuredStore()
print("📦 Registering canonical tables in DuckDB...")
duckdb_store.register_chunk_tables(rag_ready_chunks)

### Steps 3.3 and 3.4: Vector store, BM25 and unified retriever

**1. Dense retrieval.** Each chunk text is embedded with `text-embedding-3-small` into a vector $\mathbf{d} \in \mathbb{R}^n$ and L2-normalized:
$$\hat{\mathbf{d}} = \frac{\mathbf{d}}{\lVert \mathbf{d} \rVert_2 + \varepsilon}, \quad \varepsilon = 10^{-10}$$
The small $\varepsilon$ avoids division by zero. For unit vectors, cosine similarity reduces to a dot product:
$$\text{sim}(\mathbf{q}, \mathbf{d}) = \cos\theta = \hat{\mathbf{q}} \cdot \hat{\mathbf{d}}$$

**2. Sparse retrieval (BM25).** For query $Q = \{q_1, \dots, q_m\}$ and document $D$:
$$\text{BM25}(D, Q) = \sum_{i=1}^{m} \text{IDF}(q_i) \cdot \frac{f(q_i, D)\,(k_1 + 1)}{f(q_i, D) + k_1\left(1 - b + b \frac{|D|}{\text{avgdl}}\right)}$$
$$\text{IDF}(q_i) = \ln\!\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$$
where $f(q_i, D)$ is the term frequency, $|D|$ the document length, $\text{avgdl}$ the average document length, $N$ the corpus size and $n(q_i)$ the number of documents containing $q_i$. Defaults are typically $k_1 \approx 1.5$ and $b = 0.75$. BM25 excels at exact tokens such as `EBITDA` or `NRR`.

**3. Reciprocal Rank Fusion (RRF).** Scores from the two systems are not on the same scale, so only **ranks** are fused:
$$\text{RRF}(d) = \sum_{r \in \{\text{vector},\ \text{BM25}\}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$
A document ranked highly by both systems accumulates the largest score. The constant $k$ dampens the dominance of a single top rank. In the code, `rank` starts at 0, hence the `+ 1`.

**4. Metadata filtering.** `filter_ticker` restricts candidates to one company, preventing cross-company contamination (for example, Apple results in a CloudScale query).

Note: with only a handful of chunks, `top_k=5` on each retriever effectively returns all of them. The design pays off as the corpus grows.

In [ ]:
# ============================================================
# STEP 3.3 + 3.4: Vector store, BM25 and unified hybrid retriever
# ============================================================

import openai
import numpy as np
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any, Tuple, Optional

# 1. Vector storage + BM25
class HybridVectorAndSparseStore:
    """
    Handles vector storage (cosine similarity with OpenAI embeddings)
    and sparse storage (BM25 for exact matching of KPIs and terms).
    """
    def __init__(self, chunks: List[RAGReadyChunkSchema]):
        self.chunks = chunks
        self.embeddings: List[np.ndarray] = []
        self.bm25: Optional[BM25Okapi] = None
        self.corpus_tokens: List[List[str]] = []

        self._build_indexes()

    def _build_indexes(self):
        print("🧠 [Vector Store] Generating embeddings with text-embedding-3-small...")
        client = openai.OpenAI()

        # Vector indexing
        texts_to_embed = [c.vectorContent for c in self.chunks]
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=texts_to_embed
        )

        for item in response.data:
            vec = np.array(item.embedding, dtype=np.float32)
            norm_vec = vec / (np.linalg.norm(vec) + 1e-10)
            self.embeddings.append(norm_vec)

        print(f"  ✓ {len(self.embeddings)} embeddings indexed.")

        # Sparse BM25 indexing
        print("🔤 [Sparse Index] Building BM25 index...")
        for c in self.chunks:
            full_text = f"{c.vectorContent} {' '.join(c.metadata.kpisPresent)} {' '.join(c.metadata.segmentsPresent)} {c.metadata.companyName}"
            tokens = full_text.lower().replace("\n", " ").split()
            self.corpus_tokens.append(tokens)

        self.bm25 = BM25Okapi(self.corpus_tokens)
        print("  ✓ BM25 index ready.")

    def search_vector(self, query: str, top_k: int = 3) -> List[Tuple[RAGReadyChunkSchema, float]]:
        client = openai.OpenAI()
        resp = client.embeddings.create(model="text-embedding-3-small", input=[query])
        q_vec = np.array(resp.data[0].embedding, dtype=np.float32)
        q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-10)

        scores = [float(np.dot(q_vec, doc_vec)) for doc_vec in self.embeddings]
        ranked_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], scores[i]) for i in ranked_indices]

    def search_bm25(self, query: str, top_k: int = 3) -> List[Tuple[RAGReadyChunkSchema, float]]:
        q_tokens = query.lower().split()
        scores = self.bm25.get_scores(q_tokens)
        ranked_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in ranked_indices]


# 2. Hybrid search coordinator with RRF
class UnifiedHybridRetriever:
    """
    Hybrid search coordinator combining Vector + BM25 with
    Reciprocal Rank Fusion (RRF) and metadata filters (ticker, company).
    """
    def __init__(self, hybrid_store: HybridVectorAndSparseStore, duckdb_store: DuckDBStructuredStore):
        self.hybrid_store = hybrid_store
        self.duckdb_store = duckdb_store

    def hybrid_search(
        self,
        query: str,
        filter_ticker: Optional[str] = None,
        top_k: int = 2,
        rrf_k: int = 60
    ) -> List[Dict[str, Any]]:
        vec_results = self.hybrid_store.search_vector(query, top_k=5)
        bm25_results = self.hybrid_store.search_bm25(query, top_k=5)

        rrf_scores: Dict[str, float] = {}
        chunk_map: Dict[str, RAGReadyChunkSchema] = {}

        for rank, (chunk, score) in enumerate(vec_results):
            if filter_ticker and chunk.metadata.tickerOrCode.lower() != filter_ticker.lower():
                continue
            chunk_map[chunk.id] = chunk
            rrf_scores[chunk.id] = rrf_scores.get(chunk.id, 0.0) + (1.0 / (rrf_k + rank + 1))

        for rank, (chunk, score) in enumerate(bm25_results):
            if filter_ticker and chunk.metadata.tickerOrCode.lower() != filter_ticker.lower():
                continue
            chunk_map[chunk.id] = chunk
            rrf_scores[chunk.id] = rrf_scores.get(chunk.id, 0.0) + (1.0 / (rrf_k + rank + 1))

        sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        output = []
        for cid, rrf_score in sorted_ids:
            chunk = chunk_map[cid]
            output.append({
                "chunk": chunk,
                "rrf_score": rrf_score,
                "company": chunk.metadata.companyName,
                "ticker": chunk.metadata.tickerOrCode,
                "title": chunk.metadata.documentTitle
            })

        return output


# 3. Ordered instantiation
print("🚀 Initializing hybrid storage (Vector + BM25)...")
hybrid_store = HybridVectorAndSparseStore(rag_ready_chunks)

print("\n🚀 Instantiating UnifiedHybridRetriever...")
retriever = UnifiedHybridRetriever(hybrid_store, duckdb_store)
print("✅ UnifiedHybridRetriever ready to receive queries.")

### Step 3.5: Demonstration of hybrid search and SQL
Two complementary demos:

1. **Semantic hybrid search** for the question about Enterprise margin and EBITDA evolution, filtered to `PORTFOLIO_CO_04`, returning the top RRF result.
2. **Exact SQL** on `portfolio_co_04_pnl`, returning the Enterprise revenue and EBITDA rows.

**Expected verification (SQL result):**
- Enterprise revenue growth: $\dfrac{6100}{4100} - 1 \approx 48.8\%$ (Q4 2023 to Q3 2024, which the sheet reports as 0.48).
- EBITDA growth: $\dfrac{1003}{220} - 1 \approx 355.9\%$ (reported as 3.55).

These independent calculations confirm that SQL output is consistent with the narrative and reduce the risk of hallucinated figures.

In [ ]:
# ============================================================
# STEP 3.5: Hybrid search + SQL demonstration
# ============================================================

print("="*70)
print("DEMO 1: Semantic hybrid search (private portfolio)")
print("="*70)

query_1 = "How did the margin and EBITDA of the Enterprise segment evolve?"
results_1 = retriever.hybrid_search(query_1, filter_ticker="PORTFOLIO_CO_04", top_k=1)

for r in results_1:
    print(f"📌 Result found (RRF score: {r['rrf_score']:.4f})")
    print(f"   Company: {r['company']} ({r['ticker']})")
    print(f"   Document: {r['title']}")
    print(f"\n   [Retrieved content fragment]:\n{r['chunk'].vectorContent[:450]}...\n")


print("\n" + "="*70)
print("DEMO 2: Exact SQL query in DuckDB (zero hallucination)")
print("="*70)

print("Tables in DuckDB:", duckdb_store.list_tables())

# SQL query over the Board Pack table
sql_demo = """
SELECT
    "Metric / Segment",
    "Q4 2023 ($K)",
    "Q3 2024 ($K)",
    "YoY Growth (%)"
FROM portfolio_co_04_pnl
WHERE "Metric / Segment" LIKE '%Enterprise%' OR "Metric / Segment" LIKE '%EBITDA%'
"""

print(f"\nRunning SQL query:\n{sql_demo.strip()}\n")
df_sql = duckdb_store.query_sql(sql_demo)
display(df_sql)

## Phase 4: Multi-agent orchestration (LangGraph)

A sequential graph of specialized agents, each with a single responsibility:

```mermaid
flowchart LR
    U[User query] --> R[Router Agent]
    R --> A[Analyst Agent]
    A --> P[Report Agent]
    P --> O[Investment memo + risk flags]
```

Formally the workflow is a directed acyclic graph $G = (V, E)$ with $V = \{\text{router}, \text{analyst}, \text{report}\}$ and $E = \{(\text{router}, \text{analyst}), (\text{analyst}, \text{report})\}$. Each node is a function $f: S \to S$ over a shared state $S$, so the pipeline is the composition
$$S_{\text{final}} = (f_{\text{report}} \circ f_{\text{analyst}} \circ f_{\text{router}})(S_0)$$

### Step 4.1: Install LangGraph and imports
Installs `langgraph` and imports `StateGraph` and the `END` terminal node.

In [ ]:
# ============================================================
# STEP 4.1: Install LangGraph and imports
# ============================================================
!pip install langgraph -q

import openai
import json
from typing import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, END

print("✅ LangGraph and dependencies loaded for the multi-agent architecture.")

### Step 4.2: Global graph state
`AgentState` is a typed dictionary shared across nodes: user query, resolved ticker/company, `query_type`, SQL context, retrieved chunks, analysis, risk flags and the final report.

A typed shared state makes data flow explicit and debuggable. Each node reads some fields and writes others, so the state is progressively enriched: $S_0 \subset S_1 \subset S_2 \subset S_3$ in terms of populated fields.

In [ ]:
# ============================================================
# STEP 4.2: Global graph state definition
# ============================================================

class AgentState(TypedDict):
    user_query: str
    target_ticker: Optional[str]
    company_name: Optional[str]
    query_type: Optional[str]        # 'quantitative_sql' | 'qualitative_semantic' | 'full_report'
    sql_result: Optional[str]
    retrieved_chunks: List[Dict[str, Any]]
    analysis_findings: Optional[str]
    risk_flags: List[str]
    final_report: Optional[str]

print("✅ AgentState defined.")

### Step 4.3: Graph nodes
**Router Agent.** Classifies the query into a target company and an intent: `quantitative_sql`, `qualitative_semantic` or `full_report`. It runs at `temperature=0.0` for stable routing.

**Analyst Agent.** Combines two evidence sources:
- **Exact data:** `SELECT *` on the company's DuckDB tables (for quantitative and full-report intents).
- **Semantic context:** top-2 hybrid RRF chunks, filtered by ticker.

The LLM then synthesizes revenue, efficiency metrics (ARR, NRR, CAC payback, Rule of 40, EBITDA, margins), temporal comparisons and segment breakdowns, following the formulas:
$$\text{Rule of 40} = g + m, \qquad \text{Burn Multiple} = \frac{\text{Net Burn}}{\text{Net New ARR}}, \qquad \text{LTV/CAC}$$

**Report Agent.** Produces the investment-committee memo and 2-4 **risk flags** (for example margin compression, segment slowdown, cash burn, data-quality concerns). If any retrieved chunk came from an inferred schema, an **audit warning** is added.

**Known limitation to review:** `ChunkMetadataSchema` does not define `is_inferred_schema`, and the Report Agent guards the access with `hasattr`, so the audit warning will currently never trigger. Adding that field to the schema (and propagating it from the document metadata) would activate this safeguard.

**Fallbacks.** If the router returns no ticker, it defaults to `PORTFOLIO_CO_04`, a pragmatic default that could mask routing errors in production.

In [ ]:
# ============================================================
# STEP 4.3: Multi-agent graph node implementation
# ============================================================

client = openai.OpenAI()

# ------------------------------------------------------------
# NODE 1: Router / Ingestion Agent
# ------------------------------------------------------------
def router_node(state: AgentState) -> AgentState:
    """
    Identifies the target company and the user's intent.
    """
    query = state["user_query"]

    prompt = f"""
You are the Router Agent of a Private Equity platform.
Analyze the following user query and identify:
1. "target_ticker": The code/ticker of the mentioned company.
   Known options:
   - "PORTFOLIO_CO_04" (CloudScale Tech - Board Deck)
   - "PORTFOLIO_CO_07" (Apex Analytics - Auto-Documented KPIs)
   - "AAPL" (Apple Inc.)
   - "MSFT" (Microsoft Corp)
   If the company is unclear, assign "PORTFOLIO_CO_04".
2. "company_name": Formal name of the company associated with the ticker.
3. "query_type":
   - "quantitative_sql": Lookup of exact figures or tables.
   - "qualitative_semantic": Questions about strategy or explanatory summaries.
   - "full_report": Executive summary or investment committee memorandum.

User query: "{query}"

Return strictly a JSON with the structure:
{{"target_ticker": "...", "company_name": "...", "query_type": "..."}}
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    parsed = json.loads(resp.choices[0].message.content)

    state["target_ticker"] = parsed.get("target_ticker", "PORTFOLIO_CO_04")
    state["company_name"] = parsed.get("company_name", "CloudScale Tech")
    state["query_type"] = parsed.get("query_type", "full_report")

    print(f"🤖 [Router Agent] Company: {state['company_name']} ({state['target_ticker']}) | Type: {state['query_type']}")
    return state


# ------------------------------------------------------------
# NODE 2: Analyst Agent
# ------------------------------------------------------------
def analyst_node(state: AgentState) -> AgentState:
    """
    Runs relational queries in DuckDB and RRF hybrid search,
    synthesizing the key financial metrics.
    """
    ticker = state["target_ticker"]
    company = state["company_name"]
    query_type = state["query_type"]
    query = state["user_query"]

    sql_context = ""

    # 1. Exact quantitative retrieval via DuckDB
    if query_type in ["quantitative_sql", "full_report"]:
        # Map tables by ticker
        available_tables = [t for t in duckdb_store.list_tables() if ticker.lower().replace("-", "_") in t]
        if not available_tables:
            # Fallback to open search in DuckDB
            available_tables = duckdb_store.list_tables()

        sql_findings = []
        for tbl in available_tables:
            try:
                df_res = duckdb_store.query_sql(f"SELECT * FROM {tbl}")
                if not df_res.empty:
                    sql_findings.append(f"--- DuckDB table: '{tbl}' ---\n" + df_res.to_markdown(index=False))
            except Exception:
                pass
        sql_context = "\n\n".join(sql_findings) if sql_findings else "No SQL data found."

    # 2. Semantic hybrid search (Vector + BM25 via RRF)
    results = retriever.hybrid_search(query, filter_ticker=ticker, top_k=2)
    semantic_chunks = [r["chunk"].vectorContent for r in results]

    state["sql_result"] = sql_context
    state["retrieved_chunks"] = results

    # 3. Analytical synthesis with the LLM
    analyst_prompt = f"""
You are a Senior Analyst Agent at a Private Equity / Venture Capital firm.
Analyze the following information to answer the user's query rigorously.

Company: {company} ({ticker})
Query: {query}

Exact SQL data (DuckDB):
{sql_context}

Retrieved semantic context (hybrid RAG):
{chr(10).join(semantic_chunks)}

Provide a rigorous quantitative analysis, including:
- Evolution of revenue and efficiency metrics (ARR, NRR, CAC Payback, Rule of 40, EBITDA, margins) as available.
- Time comparisons (YoY / QoQ).
- Segment breakdown where applicable.
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": analyst_prompt}],
        temperature=0.1
    )

    state["analysis_findings"] = resp.choices[0].message.content
    print("📊 [Analyst Agent] Quantitative and semantic analysis generated.")
    return state


# ------------------------------------------------------------
# NODE 3: Report Agent
# ------------------------------------------------------------
def report_node(state: AgentState) -> AgentState:
    """
    Evaluates findings, identifies risk flags and generates the final memorandum.
    """
    query = state["user_query"]
    ticker = state["target_ticker"]
    company = state["company_name"]
    findings = state["analysis_findings"]

    # Check whether the source document had an inferred / auto-documented schema
    is_inferred = any(
        r["chunk"].metadata.is_inferred_schema
        for r in state.get("retrieved_chunks", [])
        if hasattr(r["chunk"].metadata, "is_inferred_schema")
    )

    audit_note = " ⚠️ [AUDIT NOTE: Data extracted from an unstructured / automatically inferred source]" if is_inferred else ""

    report_prompt = f"""
You are the Report Agent of a Private Equity / Venture Capital firm.
Your mission is to generate the final Investment Memorandum for the Investment Committee.

Company: {company} ({ticker}) {audit_note}
Original query: {query}

Analyst Agent findings:
{findings}

Generate a JSON with exactly the following structure:
1. "executive_summary": "A standardized, professional executive summary (3-4 paragraphs) covering financial performance, operating metrics and trajectory."
2. "risk_flags": ["List of 2-4 risk flags or audit observations (e.g. margin compression, segment slowdown, cash burn, or need for data verification if it comes from unstructured sources)."]

Strict JSON format:
{{"executive_summary": "...", "risk_flags": ["...", "..."]}}
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": report_prompt}],
        temperature=0.2
    )

    parsed = json.loads(resp.choices[0].message.content)
    state["risk_flags"] = parsed.get("risk_flags", [])

    audit_header = f"\n⚠️ [AUDIT WARNING]: Data schema auto-documented with an LLM." if is_inferred else ""

    final_output = f"""
======================================================================
📊 EXECUTIVE INVESTMENT MEMORANDUM
Company: {company} ({ticker}){audit_header}
======================================================================

📝 EXECUTIVE SUMMARY:
{parsed.get('executive_summary', '')}

⚠️ RISK AND AUDIT FLAGS:
{chr(10).join('🚨 ' + flag for flag in state['risk_flags'])}

----------------------------------------------------------------------
🔬 TECHNICAL DETAIL AND QUANTITATIVE ANALYSIS (Analyst Agent):
{findings}
""".strip()

    state["final_report"] = final_output
    print("🚩 [Report Agent] Investment memorandum and risk flags structured.")
    return state

print("✅ Multi-agent graph nodes ready.")

### Step 4.4: Build and compile the graph
Registers the three nodes, sets `router` as the entry point, wires `router -> analyst -> report -> END`, and compiles the workflow into an executable `app`.

Compilation validates the topology (no dangling nodes, defined entry and exit) before any LLM call is made, so structural errors surface early.

In [ ]:
# ============================================================
# STEP 4.4: Build and compile the LangGraph workflow
# ============================================================

workflow = StateGraph(AgentState)

# Register nodes
workflow.add_node("router", router_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("report", report_node)

# Define flow (edges)
workflow.set_entry_point("router")
workflow.add_edge("router", "analyst")
workflow.add_edge("analyst", "report")
workflow.add_edge("report", END)

# Compile graph
app = workflow.compile()

print("✅ Multi-agent graph compiled successfully.")

### Step 4.5: End-to-end execution (orphan-table company)
`run_pipeline` builds an empty initial state, invokes the compiled graph and prints the memo.

The test targets **Apex Analytics (Portfolio Co #7)**, whose data came from the header-less Excel dump, so it exercises the whole chain: orphan detection, LLM auto-documentation, enrichment, hybrid retrieval, analysis and reporting.

**What to verify in the output:**
- The router selects `PORTFOLIO_CO_07`.
- Metrics match the source dump (for example Rule of 40 rising $42\% \to 48\% \to 54\%$, and Burn Multiple falling $1.1\times \to 0.9\times \to 0.7\times$, which together indicate improving efficiency).
- The risk flags mention the inferred nature of the data (see the limitation noted in Step 4.3).

In [ ]:
# ============================================================
# STEP 4.5: Run on Apex Analytics (auto-documented orphan table)
# ============================================================

def run_pipeline(query: str):
    print("="*70)
    print("🚀 RUNNING PRIVATE MARKETS DOCUMENT AGENT PIPELINE")
    print(f"Query: '{query}'")
    print("="*70 + "\n")

    initial_state: AgentState = {
        "user_query": query,
        "target_ticker": None,
        "company_name": None,
        "query_type": None,
        "sql_result": None,
        "retrieved_chunks": [],
        "analysis_findings": None,
        "risk_flags": [],
        "final_report": None
    }

    result = app.invoke(initial_state)
    print("\n" + result["final_report"])

# Run on the company with the orphan table
run_pipeline(
    "Analyze the operating efficiency, ARR, Burn Multiple and SaaS metrics of Apex Analytics (Portfolio Co #7) and produce a risk report."
)